# LOF hyperparam selection

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import time
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

## Winner LOF

In [3]:
def pick_lof_params(df, consensus_threshold):

    """
    Given a dataframe of boolean anomaly flags from different
    LOF models (with different hyperparameters), calculate
    which parameter set is the most stable, i.e. which
    parameter set's anomalies overlap the most with the
    consensus anomalies (those that at least `consensus_threshold`
    models agreed were anomalies).
    """

    assert consensus_threshold > 0
    assert consensus_threshold <= df.shape[1] - 1

    stability_scores = {}

    # Identify points that at least N of the models agreed were anomalies
    consensus_anomalies = df['consensus_score'] >= consensus_threshold

    for col in df.columns[:-1]: # Exclude the consensus_score column
        # How many of this model's anomalies are also 'consensus' anomalies?
        overlap = (df[col] & consensus_anomalies).sum()
        stability_scores[col] = overlap

    # The "Optimal" params are the ones with the highest overlap
    best_params = max(stability_scores, key=stability_scores.get)

    print(f"The most stable parameter set is: {best_params}")

    return stability_scores, best_params

## Scale data

In [4]:
def standard_scaler(latent_arr):

    scaler = StandardScaler()
    
    latent_scaled = scaler.fit_transform(latent_arr)
    
    return latent_scaled

# Config

## Directories

In [5]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [6]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


## Latent per bin

In [7]:
latent_bin_dict = {}

for bin_id in bins_ids:

    latent_bin_dict[bin_id] = np.load(
        f"{latent_dir}/{bin_id}/latent_{bin_id}.npy"
    )

# Consensus based stability

In [16]:
lof_hyper_params_df_dict = {}

for bin_id in bins_ids:

    print(f"Loading LOF hyperparameter search results for {bin_id}")
    lof_hyper_params_df_dict[bin_id] = pd.read_csv(
        f"{latent_dir}/{bin_id}/lof_hypersearch_{bin_id}.csv",
    )

Loading LOF hyperparameter search results for bin_00
Loading LOF hyperparameter search results for bin_01
Loading LOF hyperparameter search results for bin_02
Loading LOF hyperparameter search results for bin_03


In [19]:
stability_scores_dict = {}
best_params_dict = {}
consensus_threshold = 8

for bin_id, df in lof_hyper_params_df_dict.items():

    print(f"Calculating stability scores for {bin_id}")

    stability_scores, best_params = pick_lof_params(
        df=df.copy(),
        consensus_threshold=consensus_threshold
    )

    stability_scores_dict[bin_id] = stability_scores
    best_params_dict[bin_id] = best_params

Calculating stability scores for bin_00
The most stable parameter set is: n60_manhattan
Calculating stability scores for bin_01
The most stable parameter set is: n60_manhattan
Calculating stability scores for bin_02
The most stable parameter set is: n60_manhattan
Calculating stability scores for bin_03
The most stable parameter set is: n60_manhattan


# Bin 4

In [22]:
consensus_03_df = lof_hyper_params_df_dict['bin_03'].copy()

In [ ]:
consensus_03_df['consensus_score'].value_counts()

consensus_score
0     176685
5       1144
10       995
1        841
2        530
3        466
4        461
9        193
8        164
6        118
7        114
15        73
11        19
14        17
12        15
13        15
Name: count, dtype: int64

In [29]:
consensus_03_df[consensus_03_df.columns[:-1]].sum(axis=1).value_counts()

0     176685
5       1144
10       995
1        841
2        530
3        466
4        461
9        193
8        164
6        118
7        114
15        73
11        19
14        17
12        15
13        15
Name: count, dtype: int64

In [25]:
stability_scores_dict['bin_03']

{'n20_euclidean': 1305,
 'n20_manhattan': 1321,
 'n20_cosine': 138,
 'n40_euclidean': 1440,
 'n40_manhattan': 1435,
 'n40_cosine': 156,
 'n60_euclidean': 1455,
 'n60_manhattan': 1462,
 'n60_cosine': 156,
 'n80_euclidean': 1442,
 'n80_manhattan': 1444,
 'n80_cosine': 155,
 'n100_euclidean': 1428,
 'n100_manhattan': 1423,
 'n100_cosine': 156}

In [32]:
consensus_threshold = 8
consensus_mask = consensus_03_df['consensus_score'] >= consensus_threshold
consensus_03_df.loc[consensus_mask, ['n60_manhattan', 'consensus_score']].sum(axis=0)


n60_manhattan       1462
consensus_score    14916
dtype: int64

## Train and save model

In [34]:
bin_id = 'bin_03'
latent_scaled = standard_scaler(
    latent_bin_dict[bin_id]
)

n, metric = best_params_dict[bin_id].split('_')[:2]
n = int(n[1:])
print(n, metric)

start_time = time.perf_counter()

model = LocalOutlierFactor(
    n_neighbors=n, 
    metric=metric, 
    contamination=0.01, 
    n_jobs=-1
)

model.fit(latent_scaled)

end_time = time.perf_counter()

print(
    f"Trained LOF model for {bin_id} "
    f"in {end_time - start_time:.2f} seconds."
)

model_fname = f"lof_{bin_id}_n{n}_{metric}.joblib"

save_path = os.path.join(
    latent_dir, bin_id, 'lof', model_fname
)

joblib.dump(model, save_path)

60 manhattan
Trained LOF model for bin_03 in 472.74 seconds.


['/home/elom/phd/code/latent/bin_03/lof/lof_bin_03_n60_manhattan.joblib']

In [35]:
bin_id= 'bin_03'
n = 60
metric = 'manhattan'
model_fname = f"lof_{bin_id}_n{n}_{metric}.joblib"

print(f"Loading model from {model_fname}")

save_path = os.path.join(
    latent_dir, bin_id, 'lof', model_fname
)
# Load the pre-trained model
loaded_lof = joblib.load(save_path)

# You can now access attributes like the negative outlier factor
# (No need to call .fit() again)
scores = loaded_lof.negative_outlier_factor_
scores.shape, scores.min(), scores.max()

Loading model from lof_bin_03_n60_manhattan.joblib


((181850,), -3.056807, -0.9540974)